## Resize `data/DATASET` to `data/archive.zip`'s worst-case ground resolution

Leaf images in `data/DATASET` were captured at roughly 0.031 cm/pixel (MicaSense RedEdge-M-style camera, 45cm tripod height). `data/archive.zip`'s drone captures (Parrot Sequoia on a senseFly eBee X) don't carry a usable height-above-ground value in their EXIF — the embedded GPS altitude there (827-849m) is absolute elevation with no ground-elevation reference to subtract, and the archive has no flight log. The worst-case ground resolution used below is therefore an **assumed** value: the eBee X's typical mapping ceiling of 120m AGL, not a number read directly from the data.

Every leaf `.tif` band is shrunk toward that coarser ground resolution with **block averaging** — each output pixel is the mean of the original pixels in its block. The `.mat` label files (FMC, chlorophyll, nitrogen, weight) are per-leaf scalars, not spatial arrays, so they're copied through unchanged.

In [1]:
import shutil
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image

### Ground sample distance: leaf dataset vs. archive.zip worst case

In [2]:
# --- Leaf dataset (data/DATASET): MicaSense RedEdge-M-style camera, tripod-mounted 45cm above the leaf ---
LEAF_PIXEL_PITCH_MM = 0.00375   # 3.75 micron
LEAF_FOCAL_LENGTH_MM = 5.5
LEAF_ALTITUDE_MM = 450          # 45 cm

LEAF_GSD_CM = (LEAF_ALTITUDE_MM * LEAF_PIXEL_PITCH_MM / LEAF_FOCAL_LENGTH_MM) / 10

# --- Drone dataset (data/archive.zip): Parrot Sequoia on a senseFly eBee X ---
# Pixel pitch and focal length below are read directly from every capture's EXIF (confirmed
# constant across all 357 captures). The altitude is NOT read from the data: EXIF GPS altitude
# (827-849m) is absolute MSL elevation with no ground-elevation reference, so it can't be turned
# into a true AGL. Worst case is instead assumed at the eBee X's typical mapping ceiling.
DRONE_PIXEL_PITCH_MM = 0.00375
DRONE_FOCAL_LENGTH_MM = 3.98
DRONE_WORST_CASE_ALTITUDE_MM = 120_000  # assumed 120m AGL

DRONE_WORST_GSD_CM = (DRONE_WORST_CASE_ALTITUDE_MM * DRONE_PIXEL_PITCH_MM / DRONE_FOCAL_LENGTH_MM) / 10

DOWNSAMPLE_FACTOR = DRONE_WORST_GSD_CM / LEAF_GSD_CM

print(f"Leaf native GSD:      {LEAF_GSD_CM:.5f} cm/px")
print(f"Drone worst-case GSD: {DRONE_WORST_GSD_CM:.5f} cm/px  (assumed {DRONE_WORST_CASE_ALTITUDE_MM/1000:.0f}m AGL)")
print(f"Downsample factor:    {DOWNSAMPLE_FACTOR:.2f}x per axis")

Leaf native GSD:      0.03068 cm/px
Drone worst-case GSD: 11.30653 cm/px  (assumed 120m AGL)
Downsample factor:    368.51x per axis


### Block-mean downsample

Block size is clamped to the image's own dimensions, so a leaf crop smaller than one block collapses to a single pixel instead of erroring out.

In [3]:
def block_mean_downsample(arr: np.ndarray, factor: float) -> np.ndarray:
    """Shrink a 2D array by averaging square blocks of `factor` pixels per side."""
    h, w = arr.shape
    block_h = max(1, min(round(factor), h))
    block_w = max(1, min(round(factor), w))
    new_h = max(1, h // block_h)
    new_w = max(1, w // block_w)

    cropped = arr[: new_h * block_h, : new_w * block_w].astype(np.float64)
    reshaped = cropped.reshape(new_h, block_h, new_w, block_w)
    return reshaped.mean(axis=(1, 3))

### Resize every leaf image

In [4]:
DATA_DIR = Path("../data") if Path("../data/DATASET").exists() else Path("data")
SRC_ROOT = DATA_DIR / "DATASET"
DST_ROOT = DATA_DIR / "DATASET_resized"

species_dirs = sorted(d for d in SRC_ROOT.iterdir() if d.is_dir())
tif_paths = []
for sp in species_dirs:
    tif_paths += sorted((sp / "Multispectral Images").glob("*.tif"))

print(f"{len(tif_paths)} images to resize across {len(species_dirs)} species")

resized_sizes = []
for src in tif_paths:
    rel = src.relative_to(SRC_ROOT)
    dst = DST_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)

    arr = np.array(Image.open(src))
    shrunk = block_mean_downsample(arr, DOWNSAMPLE_FACTOR)
    shrunk_u16 = np.clip(np.round(shrunk), 0, 65535).astype(np.uint16)

    Image.fromarray(shrunk_u16).save(dst)
    resized_sizes.append(shrunk_u16.shape)

print(f"resized {len(tif_paths)} images -> {DST_ROOT}")

7975 images to resize across 3 species


resized 7975 images -> ../data/DATASET_resized


### Carry the `.mat` labels through unchanged

FMC, chlorophyll, nitrogen, and weight are one scalar per leaf per dehydration stage — there's no pixel grid in them to shrink, so they're copied as-is alongside the resized images.

In [5]:
mat_paths = []
for sp in species_dirs:
    mat_paths += sorted(sp.glob("*.mat"))

for src in mat_paths:
    rel = src.relative_to(SRC_ROOT)
    dst = DST_ROOT / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print(f"copied {len(mat_paths)} label files unchanged (no spatial dimension to shrink)")

copied 15 label files unchanged (no spatial dimension to shrink)


### Result

In [6]:
size_counts = Counter(resized_sizes)
print("Output image size distribution (H x W):")
for size, count in sorted(size_counts.items(), key=lambda x: -x[1]):
    print(f"  {size[0]}x{size[1]} px: {count} images")

Output image size distribution (H x W):
  1x1 px: 7864 images
  1x2 px: 111 images


At a ~368x downsample factor, 7,864 of the 7,975 images (99%) collapse to a single pixel, and the rest — the widest leaf crops — reach only 1x2. This is the correct result of matching the two datasets' ground resolution, not a bug: the leaf camera resolves ~0.03cm/pixel from 45cm away, the drone resolves ~11.3cm/pixel from 120m up, a ~370x gap. `data/DATASET_resized` is useful for demonstrating that resolution-matching pipeline itself, but a single-pixel image carries only a mean reflectance value per band — it can't support anything that needs spatial structure (a CNN's convolutions, texture, leaf shape). What it *can* still feed is the same per-band regression approach as the PLSR baseline from the model reference — mean band values in, trait out — just at the drone's native resolution instead of the leaf camera's.